# Strands Agent with Datadog Observability on Amazon Bedrock AgentCore Runtime

## Overview

This notebook demonstrates deploying a Strands agent to Amazon Bedrock AgentCore Runtime with **Datadog** observability. Traces are exported to Datadog's OTLP intake via ADOT (AWS Distro for OpenTelemetry) auto-instrumentation — no Datadog Agent sidecar required.

## Key Components

- **Strands Agents**: Python framework for building LLM-powered agents with built-in telemetry support
- **Amazon Bedrock AgentCore Runtime**: Managed runtime service for hosting and scaling agents on AWS
- **Datadog**: Full-stack observability platform with APM, tracing, and AI-focused monitoring
- **ADOT**: AWS Distro for OpenTelemetry — auto-instruments your agent and exports traces via OTLP

## Prerequisites

- Python 3.10+
- AWS credentials configured with Bedrock and AgentCore permissions
- Datadog account with an API key ([free trial](https://www.datadoghq.com/free-datadog-trial/))
- Docker installed locally (or use CodeBuild for remote builds)
- Access to Amazon Bedrock Claude models in your region

## Step 1: Install Dependencies

Install the required packages from the `requirements.txt` file:

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Step 2: Set Datadog Configuration

Set your Datadog API key and site. These will be passed as OTEL environment variables to the agent runtime at deploy time.

Replace `<YOUR_DATADOG_API_KEY>` with your actual Datadog API key.

In [ ]:
import boto3
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

DD_API_KEY = ""  # Replace with your Datadog API key
DD_SITE = "datadoghq.com"  # Change if using a different Datadog site (e.g. datadoghq.eu)

## Step 3: Write the Agent Code

The agent uses Strands with a calculator tool (built-in) and a custom weather tool. Traces are handled automatically by ADOT — no manual OpenTelemetry setup needed in the agent code.

In [ ]:
%%writefile strands_datadog.py
"""Strands agent with Datadog observability on Amazon Bedrock AgentCore Runtime.

Traces are exported to Datadog via ADOT auto-instrumentation.
All OTEL config is passed as environment variables at deploy time.
"""

import os
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)


@tool
def weather() -> str:
    """Get the current weather."""
    return "sunny"


model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-20250514-v1:0")
model = BedrockModel(model_id=model_id)

agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt="You are a helpful assistant. You can do simple math calculations and tell the weather.",
)

app = BedrockAgentCoreApp()


@app.entrypoint
def handler(payload: dict, context=None) -> str:
    """Handle incoming agent invocations."""
    user_input = payload.get("prompt", "Hello!")
    logger.info("Processing prompt: %s", user_input[:100])
    response = agent(user_input)
    return response.message["content"][0]["text"]


if __name__ == "__main__":
    app.run()

## Step 4: Configure AgentCore Runtime Deployment

Use the starter toolkit to configure the AgentCore Runtime deployment. ADOT auto-instrumentation is enabled by default — it reads the OTEL environment variables we pass at deploy time to route traces to Datadog.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()
agent_name = "strands_datadog_observability"

response = agentcore_runtime.configure(
    entrypoint="strands_datadog.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
)
response

## Step 5: Deploy to AgentCore Runtime

Launch the agent with the OTEL environment variables that route traces to Datadog. ADOT reads these at container startup and auto-instruments all Bedrock model calls, tool invocations, and agent lifecycle spans.

In [ ]:
launch_result = agentcore_runtime.launch(
    auto_update_on_conflict=True,
    env_vars={
        "AGENT_OBSERVABILITY_ENABLED": "true",
        "OTEL_EXPORTER_OTLP_PROTOCOL": "http/protobuf",
        "OTEL_EXPORTER_OTLP_TRACES_ENDPOINT": f"https://otlp.{DD_SITE}/v1/traces",
        "OTEL_EXPORTER_OTLP_TRACES_HEADERS": f"dd-api-key={DD_API_KEY},dd-otlp-source=llmobs",
        "OTEL_EXPORTER_OTLP_TRACES_PROTOCOL": "http/protobuf",
        "OTEL_PYTHON_CONFIGURATOR": "aws_configurator",
        "OTEL_PYTHON_DISTRO": "aws_distro",
        "OTEL_SERVICE_NAME": "agentcore-datadog-demo",
    }
)
launch_result

## Step 6: Wait for Deployment

Poll the runtime status until the endpoint is ready.

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)

print(f"\nFinal status: {status}")

## Step 7: Invoke the Agent

Send prompts to the deployed agent. Each invocation generates traces that are exported to both CloudWatch (automatic) and Datadog (via ADOT OTLP export).

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "What is the weather now?"})

In [ ]:
from IPython.display import Markdown, display
display(Markdown("".join(invoke_response["response"])))

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "What is 42 * 17?"})

In [ ]:
display(Markdown("".join(invoke_response["response"])))

## Step 8: View Traces in Datadog

Open your Datadog APM dashboard and filter by `service:agentcore-datadog-demo`.

You should see traces for each agent invocation with spans for:
- Agent invocation lifecycle
- Model calls (Bedrock `InvokeModel`)
- Tool execution (calculator, weather)
- Token usage and latency metrics

<!-- TODO: Replace with actual screenshot -->
![Datadog APM Dashboard](images/datadog_dashboard.png)
*Datadog APM showing AgentCore agent traces with LLM and tool spans.*

## Cleanup (Optional)

Delete the AgentCore Runtime and ECR repository.

In [ ]:
# Delete the AgentCore Runtime and ECR repository
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

# Delete the runtime
agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

# Delete the ECR repository
ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split("/")[1],
    force=True,
)

print("Cleanup completed")

## Summary

You have successfully deployed a Strands agent to Amazon Bedrock AgentCore Runtime with Datadog observability. The implementation demonstrates:

- ADOT auto-instrumentation routing traces to Datadog via OTLP env vars
- Direct OTLP export to Datadog (no sidecar agent needed)
- Dual observability: CloudWatch (automatic from AgentCore) + Datadog (via ADOT)
- Deployment using the AgentCore starter toolkit

The agent is now running in a managed, scalable environment with full observability through Datadog APM.